# Control de **solo reinicio**: ¿el MoI gana por el parche o solo por cortar y volver a empezar? (Colab L4)

El brazo MoI corta la respuesta del modelo base donde el router detecta el loop y la **continúa con el modelo
parchado** (semilla nueva, castigando los tokens ya usados). En MATH-500 una **dirección al azar** continuada igual
**empató** con la nuestra (18.7 % vs 15.3 %). Sospecha: lo que ayuda es el reinicio, no el parche.

Aquí se continúan **exactamente las mismas peticiones** con el modelo base **sin parche**. Nada más cambia.

- Si `restart` ≈ `wsi_moi` → la ganancia es el **reinicio** (el detector y el momento de actuar), no la edición.
- Si `wsi_moi` > `restart` (IC 95 % arriba de 0) → la edición sí aporta.

Reusa dos corridas ya terminadas en Drive (no repite nada de ellas):
1. **MATH-500** 0.8B: `wsi_runs/discover_math500_qwen35-08b` (150 problemas, 16k).
2. **GSM8K** 0.8B: `wsi_runs/discover_blind` (200 problemas, 16k), donde la dirección **sí** le ganó al azar
   (55.5 % vs 32 %). Si esa carpeta no está en tu Drive, se salta.

~45–70 min en L4. Resultado: `MyDrive/wsi_runs/FINALES/restart_control_qwen35-08b.json`. Si se corta, `Run all`
otra vez y retoma. Al final el runtime se apaga solo.

In [ ]:
MODEL = "Qwen/Qwen3.5-0.8B"
RUN_NAME = "restart_control_qwen35-08b"
RUNS = {"math": "discover_math500_qwen35-08b", "gsm8k": "discover_blind"}   # carpetas en MyDrive/wsi_runs
REPO = "/content/wsi"
BASE = "/content/base"
S = f"{REPO}/scripts/restart_control.py"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm sympy antlr4-python3-runtime==4.11.1 hf_transfer
!pip install -q flash-linear-attention || echo "fla opcional"
import vllm, transformers; print("vllm", vllm.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, glob, shutil, json, subprocess, datetime
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/wsi_runs"
FINALES = f"{ROOT}/FINALES"
OUTD = f"{ROOT}/{RUN_NAME}"
os.makedirs(FINALES, exist_ok=True)
WORK = {}
for ds, run in RUNS.items():
    src = f"{ROOT}/{run}"
    if not os.path.exists(f"{src}/requests_test_moi.jsonl"):
        print(f"{ds}: no encuentro {src}/requests_test_moi.jsonl -> se salta")
        continue
    w, keep = f"/content/work_{ds}", f"{OUTD}/{ds}"
    os.makedirs(w, exist_ok=True); os.makedirs(keep, exist_ok=True)
    for f in glob.glob(f"{src}/*.json") + glob.glob(f"{src}/*.jsonl"):   # la corrida original (solo lectura)
        if not os.path.exists(f"{w}/{os.path.basename(f)}"):
            shutil.copy2(f, w)
    for f in glob.glob(f"{keep}/*"):                                       # lo que esta corrida ya hizo
        if os.path.isfile(f):
            shutil.copy2(f, w)
    WORK[ds] = w
    print(ds, "<-", src)
assert WORK, "no hay ninguna corrida que reusar en Drive"

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git fetch -q --depth 1 origin main && git reset -q --hard origin/main
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert os.path.exists(S), "el repo no tiene restart_control.py: falta push"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)

In [ ]:
import time
# a vLLM engine left over from an interrupted cell keeps the GPU busy and the next run hangs: kill it first
!nvidia-smi --query-compute-apps=pid --format=csv,noheader | xargs -r kill -9

def run_step(cmd, env):
    print("\n>>>", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
    buf, last = b"", 0.0
    while True:
        ch = p.stdout.read1(4096)
        if not ch:
            break
        buf += ch
        *lines, buf = buf.replace(b"\r\n", b"\n").replace(b"\r", b"\n").split(b"\n")
        for l in lines:
            s = l.decode(errors="replace").strip()
            if not s or "Attempted to load weight" in s:
                continue
            if "it/s" in s or "s/it" in s:  # tqdm: one line every 30 s
                if time.time() - last < 30:
                    continue
                last = time.time()
            print(s, flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Step failed: {' '.join(cmd)}")

RES = {}
for ds, w in WORK.items():
    keep = f"{OUTD}/{ds}"
    # resultados nuevos a una carpeta propia: la corrida original en Drive no se toca
    env = {**os.environ, "WSI_SYNC_DIR": keep, "PYTHONPATH": f"{REPO}/scripts:{REPO}"}
    run_step(["python", "-u", S, "gen", "--work", w, "--base-dir", BASE], env)
    shutil.copy2(f"{w}/outputs_test_moi_restart.jsonl", keep)
    run_step(["python", "-u", S, "report", "--work", w, "--dataset", ds], env)
    RES[ds] = json.load(open(f"{w}/results_restart_control.json"))
    shutil.copy2(f"{w}/results_restart_control.json", keep)

In [ ]:
out = {"runs": RES, "reused": {ds: RUNS[ds] for ds in RES}}
for ds, r in RES.items():
    a = r["arms"]
    print(f"{ds}: base {a['base']['correct']:.1%} | restart {a.get('restart_moi', {}).get('correct', 0):.1%} | "
          f"wsi_moi {a.get('wsi_moi', {}).get('correct', 0):.1%} | random_moi {a.get('random_moi', {}).get('correct', 0):.1%}"
          f"  -> {r.get('verdict')}")
out["_final"] = {"complete": True, "model": MODEL, "commit": COMMIT, "notebook": "restart_control_colab.ipynb",
                 "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
                 "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as fh:
    json.dump(out, fh, indent=1)
os.replace(dst + ".tmp", dst)
print("\nGuardado final en Drive:", dst)

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    print("Resultado en FINALES. Apagando el runtime en 60 s...", flush=True)
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")